### Imports & Preparations

In [1]:
import numpy as np
import torch
from torchvision.datasets import CIFAR100
import altair as alt
import pandas as pd
from sklearn.manifold import MDS

alt.data_transformers.enable("vegafusion");

In [2]:
dataset = CIFAR100(root="./data", train=False, download=True)
class_names = dataset.classes
emb_images = np.load("./data/cifar100_test_embeddings.npy") # (image, embedding)
true_classes = np.load("./data/cifar100_test_labels.npy") # (image,)
label_data = np.load("./data/cifar100_text_embeddings.npz")
emb_labels = label_data["embeddings"] # (template, class, embedding)
label_templates = [str(i) for i in label_data["templates"]]

In [3]:
# similar classes are placed roughly nearby
CLASS_ORDER = [
    "beaver", "dolphin", "otter", "seal", "whale",
    "aquarium_fish", "flatfish", "ray", "shark", "trout",
    "bear", "leopard", "lion", "tiger", "wolf",
    "camel", "cattle", "chimpanzee", "elephant", "kangaroo",
    "fox", "porcupine", "possum", "raccoon", "skunk",
    "hamster", "mouse", "rabbit", "shrew", "squirrel",
    "crocodile", "dinosaur", "lizard", "snake", "turtle",
    "bee", "beetle", "butterfly", "caterpillar", "cockroach",
    "crab", "lobster", "snail", "spider", "worm",
    "baby", "boy", "girl", "man", "woman",
    "orchid", "poppy", "rose", "sunflower", "tulip",
    "maple_tree", "oak_tree", "palm_tree", "pine_tree", "willow_tree",
    "apple", "mushroom", "orange", "pear", "sweet_pepper",
    "cloud", "forest", "mountain", "plain", "sea",
    "bottle", "bowl", "can", "cup", "plate",
    "clock", "keyboard", "lamp", "telephone", "television",
    "bed", "chair", "couch", "table", "wardrobe",
    "bicycle", "bus", "motorcycle", "pickup_truck", "train",
    "lawn_mower", "rocket", "streetcar", "tank", "tractor",
    "bridge", "castle", "house", "road", "skyscraper",
]
assert set(CLASS_ORDER) == set(class_names)

In [4]:
image_label_sim = np.einsum("ie,tce->itc", emb_images, emb_labels) # (image, template, class)

### Top-3 Accuracy by Template and Class

In [5]:
k = 3
topk_indices = np.argpartition(image_label_sim, axis=2, kth=-k)[:, :, -k:]
successes = (topk_indices == true_classes[:, None, None]).any(axis=2) # (image, template)
accuracies = np.array([successes[true_classes == i].mean(axis=0) for i in range(len(class_names))]) # (class, template)
df_acc = pd.DataFrame({
    "class": np.repeat(class_names, len(label_templates)),
    "template": np.tile(label_templates, len(class_names)),
    "accuracy": accuracies.ravel()
})
alt.Chart(df_acc).mark_circle(size=40).encode(
    x=alt.X("class:N", sort="y"), y="accuracy", color="template",
    tooltip=["template", "class", "accuracy"]
).interactive()

alt.Chart(...)

### Top-1 Confusion matrix

In [6]:
predicted = image_label_sim.argmax(axis=2) # (image, template)
counts = np.array([np.bincount(
    predicted[true_classes == i].ravel(), minlength=len(class_names)
) for i in range(len(class_names))])
confusion = counts / counts.sum(axis=1, keepdims=True)
df_confusion = pd.DataFrame({
    "true class": np.repeat(class_names, len(class_names)),
    "predicted class": np.tile(class_names, len(class_names)),
    "fraction": confusion.ravel()
})
alt.Chart(df_confusion).mark_rect().encode(
    x=alt.X("predicted class:N", sort=CLASS_ORDER), y=alt.Y("true class:N", sort=CLASS_ORDER),
    color="fraction", tooltip=["true class", "predicted class", "fraction"]
).interactive()

alt.Chart(...)

### Projections for most confused classes

In [7]:
off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
conf_symm = np.triu(off_diagonal + off_diagonal.T)
top_confused = np.argsort(conf_symm, axis=None)[::-1][:3]
pairs = [np.unravel_index(i, conf_symm.shape) for i in top_confused]
selected_classes = sorted({int(c) for p in pairs for c in p})

image_accuracy = (predicted == true_classes[:, None]).mean(axis=1)
n_good = n_bad = 20
img_items = []
for c in selected_classes:
    idx = np.flatnonzero(true_classes == c)
    by_acc = idx[np.argsort(image_accuracy[idx], kind="stable")]
    for i in [i for i in by_acc[:n_bad] if image_accuracy[i] < 1]:
        img_items.append(("image inaccurate", int(i), c))
    for i in [i for i in by_acc[::-1][:n_good] if image_accuracy[i] > 0]:
        img_items.append(("image accurate", int(i), c))
label_role = {(t, c, label_templates[t].format(class_names[c]), c): "label correct" for c in selected_classes for t in range(len(label_templates))}
for kind, i, c in img_items:
    if kind == "image inaccurate":
        for t in np.flatnonzero(predicted[i] != c):
            key = (int(t), c, label_templates[int(t)].format(class_names[int(predicted[i, t])]), int(predicted[i, t]))
            if key not in label_role:
                label_role[key] = "label inaccurate"

embs, meta = [], []
img_mean = emb_images.mean(axis=0)
label_mean = emb_labels.reshape(-1, emb_labels.shape[-1]).mean(axis=0)
for kind, i, c in img_items:
    embs.append(emb_images[i] - img_mean)
    meta.append({"kind": kind, "image_id": i, "label": None, "class": class_names[c]})
for (t,c, l, c_pred), role in label_role.items():
    embs.append(emb_labels[t, c_pred] - label_mean)
    meta.append({"kind": role, "image_id": None, "label": l, "class": class_names[c]})
meta = pd.DataFrame(meta)
E = np.stack(embs)
E = E / np.linalg.norm(E, axis=-1, keepdims=True)
S = np.clip(E @ E.T, -1, 1)
S = (S + S.T) / 2
D = np.sqrt(2 * (1-S))
np.fill_diagonal(D, 0)
xy = MDS(n_components=2, dissimilarity="precomputed", random_state=0, n_init=8).fit_transform(D)
meta[["x", "y"]] = xy

C:\Users\Mykhailo\Documents\JKU\AI and Visualization\VisforAI26_Bumblebee\.conda\Lib\site-packages\sklearn\manifold\_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
C:\Users\Mykhailo\Documents\JKU\AI and Visualization\VisforAI26_Bumblebee\.conda\Lib\site-packages\sklearn\manifold\_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


In [8]:
classes = sorted(meta["class"].unique())
pick = alt.selection_point(
    fields=["class"],
    value=[{"class": classes[0]}],                  # initial selection
    bind=alt.binding_select(options=classes, name="Class: "),
)
pad = 0.1
xdom = [meta["x"].min() - pad, meta["x"].max() + pad]
ydom = [meta["y"].min() - pad, meta["y"].max() + pad]
alt.Chart(meta).mark_point(filled=True, size=110, opacity=0.8).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=xdom)),
    y=alt.Y("y:Q", scale=alt.Scale(domain=ydom)),
    color="kind:N",
    shape="kind:N",
    tooltip=["kind", "class", "label", "image_id"],
    ).add_params(pick).transform_filter(pick).properties(width=500, height=500)

alt.Chart(...)